In [1]:
import os
import json

# Step 1: Create a .kaggle directory
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)


api_token = { # Step 2: Save your credentials
    "username":"suchitachandekar",
    "key":"dbea6394a91def5f15ac4426de7725c0"
}

with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(api_token, f)


try:     # Step 3: Set permissions (only for Unix/Linux/macOS; skip on Windows)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
except:
    pass

print(" Kaggle API key saved successfully!")


 Kaggle API key saved successfully!


In [2]:

# Step 4: Download and unzip PneumoniaMNIST dataset
!kaggle datasets download -d rijulshr/pneumoniamnist -p ./data --unzip


Dataset URL: https://www.kaggle.com/datasets/rijulshr/pneumoniamnist
License(s): MIT


In [3]:
import numpy as np

# Load the NPZ file
data = np.load('./data/pneumoniamnist.npz')  # this will find the file from the all storage

# View contents
print(data.files)


['train_images', 'train_labels', 'val_images', 'val_labels', 'test_images', 'test_labels']


In [4]:
# Extract arrays
x_train = data['train_images']
y_train = data['train_labels']
x_val = data['val_images']
y_val = data['val_labels']
x_test = data['test_images']
y_test = data['test_labels']

# Resize to (224, 224, 3) — required for InceptionV3
import tensorflow as tf

def preprocess_images(x):
    x = np.stack([x]*3, axis=-1)  # grayscale to 3 channels
    x = tf.image.resize(x, [224, 224]).numpy()
    x = x / 255.0
    return x

x_train = preprocess_images(x_train)
x_val = preprocess_images(x_val)
x_test = preprocess_images(x_test)

print(x_train.shape, x_val.shape, x_test.shape) 



(3882, 224, 224, 3) (524, 224, 224, 3) (624, 224, 224, 3)


In [5]:
x_train

array([[[[0.36078432, 0.36078432, 0.36078432],
         [0.36078432, 0.36078432, 0.36078432],
         [0.36078432, 0.36078432, 0.36078432],
         ...,
         [0.02352941, 0.02352941, 0.02352941],
         [0.02352941, 0.02352941, 0.02352941],
         [0.02352941, 0.02352941, 0.02352941]],

        [[0.36078432, 0.36078432, 0.36078432],
         [0.36078432, 0.36078432, 0.36078432],
         [0.36078432, 0.36078432, 0.36078432],
         ...,
         [0.02352941, 0.02352941, 0.02352941],
         [0.02352941, 0.02352941, 0.02352941],
         [0.02352941, 0.02352941, 0.02352941]],

        [[0.36078432, 0.36078432, 0.36078432],
         [0.36078432, 0.36078432, 0.36078432],
         [0.36078432, 0.36078432, 0.36078432],
         ...,
         [0.02352941, 0.02352941, 0.02352941],
         [0.02352941, 0.02352941, 0.02352941],
         [0.02352941, 0.02352941, 0.02352941]],

        ...,

        [[0.6784314 , 0.6784314 , 0.6784314 ],
         [0.6784314 , 0.6784314 , 0.6784314 ]

In [6]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

# Create model
base_model = InceptionV3(include_top=False, weights='imagenet', input_tensor=Input(shape=(224, 224, 3)))
base_model.trainable = False  # Freeze base

# Add classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

# Train
model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=5, batch_size=32)




Epoch 1/5


122/122 [==============================] - 245s 2s/step - loss: 0.2263 - accuracy: 0.9122 - val_loss: 0.3655 - val_accuracy: 0.8435
Epoch 2/5
122/122 [==============================] - 211s 2s/step - loss: 0.1566 - accuracy: 0.9356 - val_loss: 0.2613 - val_accuracy: 0.8855
Epoch 3/5
122/122 [==============================] - 258s 2s/step - loss: 0.1365 - accuracy: 0.9441 - val_loss: 0.2266 - val_accuracy: 0.9103
Epoch 4/5
122/122 [==============================] - 207s 2s/step - loss: 0.1271 - accuracy: 0.9523 - val_loss: 0.2803 - val_accuracy: 0.8989
Epoch 5/5
122/122 [==============================] - 192s 2s/step - loss: 0.1237 - accuracy: 0.9495 - val_loss: 0.2420 - val_accuracy: 0.9122


In [7]:
# Evaluate
model.evaluate(x_test, y_test)

# Predict
y_pred = (model.predict(x_test) > 0.5).astype("int32")

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


20/20 [==============================] - 30s 1s/step
              precision    recall  f1-score   support

           0       0.97      0.36      0.52       234
           1       0.72      0.99      0.83       390

    accuracy                           0.75       624
   macro avg       0.84      0.68      0.68       624
weighted avg       0.81      0.75      0.72       624

[[ 84 150]
 [  3 387]]


In [8]:

import cv2

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model([model.inputs],
                                       [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, model, last_conv_layer_name='mixed10', alpha=0.4):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224), color_mode='grayscale')
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.stack([img_array]*3, axis=-1)
    img_array = tf.image.resize(img_array, [224, 224]).numpy() / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
    img = cv2.resize(img_array[0].astype("uint8"), (224, 224))
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed_img = heatmap * alpha + img
    superimposed_img = np.uint8(superimposed_img)

    plt.figure(figsize=(6, 6))
    plt.imshow(superimposed_img)
    plt.axis('off')
    plt.title("Grad-CAM")
    plt.show()

# Example:
# replace the image path with yours imagae location path.
# display_gradcam("path/to/image.jpg", model)   # this will diplay the where is the pneumonia present in the X-ray
# If you want only a prediction, use 
# PREDICTION = model.predict('your image location path')

In [ ]:
# STEP 4: Save model in pickle format
import pickle
model.save("inception_model.h5")  # backup
with open("inception_model.pkl", "wb") as f:
    pickle.dump(model, f)